<a href="https://colab.research.google.com/github/Uma-Energy/SPE-Africa-DSEATS-Datathon/blob/main/UMA_MIRACLE_IDIKA_TECHRISE_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **UMA MIRACLE IDIKA - SPE AFRICA DSEATS DATATHON**

---

In this contest project, our goal is to use machine learning to classify 20 oil wells based on their production data and trends. Specifically, we will analyze daily production records (oil, gas, water, pressures, temperatures, choke sizes, etc.) to determine for each well:


1.   Reservoir type (saturated or undersaturated)
2.   Well type (Gas‑lifted vs Naturally flowing)
3.   Production behaviour (Steady vs Unsteady)
4.   GOR trend relative to solution GOR (above / below / combo)
5.   Water‑cut trend (Flat / Increasing / Decreasing / Combo)
6.   Productivity‑index trend (Flat / Increasing / Decreasing / Combo)

Finally, we will calculate the total reservoir barrels of oil produced from each reservoir, and build a machine learning model to classify the wells simmultaneously.

---


## **Let's Get Started!!**

## Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import matplotlib.pyplot as plt
import random
from datetime import timedelta
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.utils import to_categorical
import tensorflow.keras.backend as K
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping

import warnings

warnings.filterwarnings("ignore", category=UserWarning, module='sklearn')

## Load and Clean Datasets

In [ ]:
# Load Datasets
wells_filepath = '/content/drive/MyDrive/SPE DSEATS 2025/spe_africa_dseats_datathon_2025_wells_dataset.csv'
reservoir_info_filepath = '/content/drive/MyDrive/SPE DSEATS 2025/reservoir_info.csv'
classification_params_filepath = '/content/drive/MyDrive/SPE DSEATS 2025/classification_parameters.csv'

wells_df = pd.read_csv(wells_filepath)
reservoir_df = pd.read_csv(reservoir_info_filepath)
params_df = pd.read_csv(classification_params_filepath)

In [ ]:
reservoir_df

,Reservoir Name,Initial Reservoir Pressure (PSI),Bubble Point Pressure (PSI),Current Average Reservoir Pressure (PSI),Solution Gas-Oil-Ratio (SCF/BBL),Formation Volume Factor (RB/STB)
0,ACHI,"3,500","3,300","2,700",800,1.20
1,KEMA,"4,200","4,000","3,900",600,1.45
2,MAKO,"3,500","3,500","3,000",500,1.15
3,DEPU,"2,800","2,800","2,400","1,200",1.37
4,JANI,"4,500","4,300","4,200","1,000",1.30


In [ ]:
params_df

,Reservoir Name,Reservoir Type,Well Type,Production Type,Formation GOR Trend,Watercut Trend,Oil Productivity Index Trend
0,ACHI,Saturated,NF,Steady,aSolGOR,Flat,Flat
1,KEMA,Undersat,GL,Unsteady,bSolGOR,Incr,Incr
2,MAKO,NaN,NaN,NaN,Combo,Decr,Decr
3,DEPU,NaN,NaN,NaN,NaN,Combo,Combo
4,JANI,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
wells_df

,PROD_DATE,WELL_NAME,ON_STREAM_HRS,BOTTOMHOLE_FLOWING_PRESSURE (PSI),DOWNHOLE_TEMPERATURE (deg F),ANNULUS_PRESS (PSI),CHOKE_SIZE (%),WELL_HEAD_PRESSURE (PSI),WELL_HEAD_TEMPERATURE (deg F),CUMULATIVE_OIL_PROD (STB),CUMULATIVE_FORMATION_GAS_PROD (MSCF),CUMULATIVE_TOTAL_GAS_PROD (MSCF),CUMULATIVE_WATER_PROD (BBL)
0,15-Feb-14,Well_#1,0.00,"4,050",189.866,0,1.17951,482.46,50.864,0,0,0,0
1,16-Feb-14,Well_#1,0.00,"3,961",189.945,0,2.99440,328.601,47.668,0,0,0,0
2,17-Feb-14,Well_#1,0.00,"3,961",190.004,0,1.90349,387.218,48.962,0,0,0,0
3,18-Feb-14,Well_#1,0.00,"3,964",190.020,0,0.00000,308.98,46.636,0,0,0,0
4,19-Feb-14,Well_#1,0.00,"3,965",190.107,0,30.20760,196.057,47.297,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7950,05-Apr-15,Well_#20,18.94,"2,505",149.177,633.188,77.32078,115.406,85.532,"497,425","235,131","352,697","522,788"
7951,06-Apr-15,Well_#20,21.06,"2,503",149.169,617.591,72.40304,116.285,84.959,"497,609","235,205","352,808","523,266"
7952,07-Apr-15,Well_#20,24.00,"2,481",149.175,645.435,100.00000,111.943,87.361,"497,879","235,314","352,971","523,885"
7953,08-Apr-15,Well_#20,15.94,"2,485",149.178,651.282,76.40842,111.962,87.583,"498,019","235,370","353,055","524,431"


In [ ]:
wells_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7955 entries, 0 to 7954
Data columns (total 13 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   PROD_DATE                             7955 non-null   object 
 1   WELL_NAME                             7955 non-null   object 
 2   ON_STREAM_HRS                         7955 non-null   float64
 3   BOTTOMHOLE_FLOWING_PRESSURE (PSI)     7955 non-null   object 
 4   DOWNHOLE_TEMPERATURE (deg F)          7955 non-null   float64
 5   ANNULUS_PRESS (PSI)                   7955 non-null   object 
 6   CHOKE_SIZE (%)                        7955 non-null   float64
 7   WELL_HEAD_PRESSURE (PSI)              7955 non-null   object 
 8   WELL_HEAD_TEMPERATURE (deg F)         7955 non-null   float64
 9   CUMULATIVE_OIL_PROD (STB)             7955 non-null   object 
 10  CUMULATIVE_FORMATION_GAS_PROD (MSCF)  7955 non-null   object 
 11  CUMULATIVE_TOTAL_

There are no null values in the dataset. However, we can notice that some columns have wrong data types, such as the `BOTTOMHOLE_FLOWING_PRESSURE (PSI)`, `ANNULUS_PRESS (PSI)`, `WELL_HEAD_PRESSURE (PSI)`, `CUMULATIVE_OIL_PROD (STB)`, `CUMULATIVE_FORMATION_GAS_PROD (MSCF)`, `CUMULATIVE_TOTAL_GAS_PROD (MSCF)`, and `CUMULATIVE_WATER_PROD (BBL)`. This is because of the comma used in their entries. We will need to remove the commas and convert these columns to float data types. In addition, we will need to convert out `PROD_DATE` column to a datetime object.


In [ ]:
cols_to_convert = ['BOTTOMHOLE_FLOWING_PRESSURE (PSI)', 'ANNULUS_PRESS (PSI)', 'WELL_HEAD_PRESSURE (PSI)',
                   'CUMULATIVE_OIL_PROD (STB)', 'CUMULATIVE_FORMATION_GAS_PROD (MSCF)', 'CUMULATIVE_TOTAL_GAS_PROD (MSCF)', 'CUMULATIVE_WATER_PROD (BBL)'
]
for col in cols_to_convert:
    wells_df[col] = wells_df[col].str.replace(',', '').astype(float)
wells_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7955 entries, 0 to 7954
Data columns (total 13 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   PROD_DATE                             7955 non-null   object 
 1   WELL_NAME                             7955 non-null   object 
 2   ON_STREAM_HRS                         7955 non-null   float64
 3   BOTTOMHOLE_FLOWING_PRESSURE (PSI)     7955 non-null   float64
 4   DOWNHOLE_TEMPERATURE (deg F)          7955 non-null   float64
 5   ANNULUS_PRESS (PSI)                   7955 non-null   float64
 6   CHOKE_SIZE (%)                        7955 non-null   float64
 7   WELL_HEAD_PRESSURE (PSI)              7955 non-null   float64
 8   WELL_HEAD_TEMPERATURE (deg F)         7955 non-null   float64
 9   CUMULATIVE_OIL_PROD (STB)             7955 non-null   float64
 10  CUMULATIVE_FORMATION_GAS_PROD (MSCF)  7955 non-null   float64
 11  CUMULATIVE_TOTAL_

In [ ]:
# Convert 'PROD_DATE' to datetime object
wells_df['PROD_DATE'] = pd.to_datetime(wells_df['PROD_DATE'], format='%d-%b-%y')
wells_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7955 entries, 0 to 7954
Data columns (total 13 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   PROD_DATE                             7955 non-null   datetime64[ns]
 1   WELL_NAME                             7955 non-null   object        
 2   ON_STREAM_HRS                         7955 non-null   float64       
 3   BOTTOMHOLE_FLOWING_PRESSURE (PSI)     7955 non-null   float64       
 4   DOWNHOLE_TEMPERATURE (deg F)          7955 non-null   float64       
 5   ANNULUS_PRESS (PSI)                   7955 non-null   float64       
 6   CHOKE_SIZE (%)                        7955 non-null   float64       
 7   WELL_HEAD_PRESSURE (PSI)              7955 non-null   float64       
 8   WELL_HEAD_TEMPERATURE (deg F)         7955 non-null   float64       
 9   CUMULATIVE_OIL_PROD (STB)             7955 non-null   float64       
 10  

In [ ]:
wells_df

,PROD_DATE,WELL_NAME,ON_STREAM_HRS,BOTTOMHOLE_FLOWING_PRESSURE (PSI),DOWNHOLE_TEMPERATURE (deg F),ANNULUS_PRESS (PSI),CHOKE_SIZE (%),WELL_HEAD_PRESSURE (PSI),WELL_HEAD_TEMPERATURE (deg F),CUMULATIVE_OIL_PROD (STB),CUMULATIVE_FORMATION_GAS_PROD (MSCF),CUMULATIVE_TOTAL_GAS_PROD (MSCF),CUMULATIVE_WATER_PROD (BBL)
0,2014-02-15,Well_#1,0.00,4050.0,189.866,0.000,1.17951,482.460,50.864,0.0,0.0,0.0,0.0
1,2014-02-16,Well_#1,0.00,3961.0,189.945,0.000,2.99440,328.601,47.668,0.0,0.0,0.0,0.0
2,2014-02-17,Well_#1,0.00,3961.0,190.004,0.000,1.90349,387.218,48.962,0.0,0.0,0.0,0.0
3,2014-02-18,Well_#1,0.00,3964.0,190.020,0.000,0.00000,308.980,46.636,0.0,0.0,0.0,0.0
4,2014-02-19,Well_#1,0.00,3965.0,190.107,0.000,30.20760,196.057,47.297,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7950,2015-04-05,Well_#20,18.94,2505.0,149.177,633.188,77.32078,115.406,85.532,497425.0,235131.0,352697.0,522788.0
7951,2015-04-06,Well_#20,21.06,2503.0,149.169,617.591,72.40304,116.285,84.959,497609.0,235205.0,352808.0,523266.0
7952,2015-04-07,Well_#20,24.00,2481.0,149.175,645.435,100.00000,111.943,87.361,497879.0,235314.0,352971.0,523885.0
7953,2015-04-08,Well_#20,15.94,2485.0,149.178,651.282,76.40842,111.962,87.583,498019.0,235370.0,353055.0,524431.0


## Exploratory Data Analysis (EDA)

We have cleaned the data types. Next, we will delve into performing exploratory data analysis on the wells.

In [ ]:
# Get the number of unique wells
num_unique_wells = wells_df['WELL_NAME'].nunique()
print(f"Number of unique wells: {num_unique_wells}")

Number of unique wells: 20


In [ ]:
# Get the summary statistics of the dataset
wells_df.describe().T

,count,mean,min,25%,50%,75%,max,std
PROD_DATE,7955,2013-08-26 16:25:49.516027648,2011-02-17 00:00:00,2012-08-02 00:00:00,2013-06-25 00:00:00,2014-10-01 00:00:00,2016-08-12 00:00:00,NaN
ON_STREAM_HRS,7955.0,21.623497,0.0,24.0,24.0,24.0,25.0,6.567826
BOTTOMHOLE_FLOWING_PRESSURE (PSI),7955.0,2613.811816,0.0,2195.0,2465.0,3067.0,4096.0,687.60064
DOWNHOLE_TEMPERATURE (deg F),7955.0,168.757429,0.0,150.2285,158.624,202.6,212.153,31.917787
ANNULUS_PRESS (PSI),7955.0,471.339542,0.0,4.199,301.247,981.329,1639.04,481.63834
CHOKE_SIZE (%),7955.0,56.155295,0.0,28.130855,51.06803,99.80095,100.0,34.975408
WELL_HEAD_PRESSURE (PSI),7955.0,212.574151,0.0,68.6615,113.718,161.1765,1787.76,282.193044
WELL_HEAD_TEMPERATURE (deg F),7955.0,86.836365,0.0,80.6775,88.364,96.7705,182.157,22.738615
CUMULATIVE_OIL_PROD (STB),7955.0,172585.46939,0.0,47691.0,109295.0,242488.5,1129301.0,174082.717158
CUMULATIVE_FORMATION_GAS_PROD (MSCF),7955.0,145623.481961,0.0,38541.5,101610.0,199103.5,1458660.0,172168.475676


In [ ]:
# Visualize the number of data points for each well
fig = px.histogram(wells_df, x='WELL_NAME',
                   color='WELL_NAME',
                   title='Number of Data Points in Each Well')
fig.update_layout(bargap=0.2, xaxis_title='WELL_NAME', yaxis_title='Count')
fig.show()

In [ ]:
# Plot the cumulative production profiles for each well
well_names = wells_df['WELL_NAME'].unique()

for well_name in well_names:
    well_data = wells_df[wells_df['WELL_NAME'] == well_name]

    fig = go.Figure()

    # Cum. Oil Prod.
    fig.add_trace(go.Scatter(
        x=well_data['PROD_DATE'],
        y=well_data['CUMULATIVE_OIL_PROD (STB)'],
        mode='lines',
        line=dict(color='black'),
        name='Cumulative Oil Production',
        hovertemplate='Date: %{x}<br>Cum. Oil Prod.: %{y}<extra></extra>'
    ))

    # Cum. Formation Gas Prod.
    fig.add_trace(go.Scatter(
        x=well_data['PROD_DATE'],
        y=well_data['CUMULATIVE_FORMATION_GAS_PROD (MSCF)'],
        mode='lines+markers',
        line=dict(color='darkgrey'),
        name='Cumulative Formation Gas Production',
        hovertemplate='Date: %{x}<br>Cum. Fm. Gas Prod.: %{y}<extra></extra>'
    ))

    # Cum. Total Gas Prod.
    fig.add_trace(go.Scatter(
        x=well_data['PROD_DATE'],
        y=well_data['CUMULATIVE_TOTAL_GAS_PROD (MSCF)'],
        mode='lines',
        line=dict(color='red'),
        name='Cumulative Total Gas Production',
        hovertemplate='Date: %{x}<br>Cum. Total Gas Prod.: %{y}<extra></extra>'
    ))

    # Cum. Water Prod.
    fig.add_trace(go.Scatter(
        x=well_data['PROD_DATE'],
        y=well_data['CUMULATIVE_WATER_PROD (BBL)'],
        mode='lines',
        line=dict(color='blue'),
        name='Cumulative Water Production',
        hovertemplate='Date: %{x}<br>Cum. Water Prod.: %{y}<extra></extra>'
    ))

    fig.update_layout(
        title=f'Cumulative Production Profiles for {well_name}',
        xaxis_title='Date',
        yaxis_title='Cumulative Production',
        height=500
    )

    fig.show()


In [ ]:
# Plot Annulus pressure over time
for well_name in well_names:
    # Filter data for the current well
    well_data = wells_df[wells_df['WELL_NAME'] == well_name].sort_values('PROD_DATE').copy()

    # Create a Plotly figure
    fig = go.Figure()

    # Add Annulus Pressure trace
    fig.add_trace(go.Scatter(
        x=well_data['PROD_DATE'],
        y=well_data['ANNULUS_PRESS (PSI)'],
        mode='lines',
        name='Annulus Pressure',
        hovertemplate='Date: %{x}<br>Annulus Pressure: %{y:.2f} PSI<extra></extra>'
    ))

    # Update layout
    fig.update_layout(
        title=f'Annulus Pressure Profile for {well_name}',
        xaxis_title='Date',
        yaxis_title='Annulus Pressure (PSI)',
        height=500
    )

    # Show the plot
    fig.show()

For some of the wells, the annulus pressure values are above the normal range, which typically lies close to atmospheric pressure (0 to 30 psi). This indicates that the annuli are pressurized, which could be a possible sign of gas lift operations.

## **Feature Engineering**
---

From the description of wells, it is evident that there are days with zero onstream hours. These entries will be removed from the dataframe at various points in the script. Since the onstream hours vary each day, calculating the daily rates accurately requires adjusting for this variation. For each well $i$ on day $t$, the equation for calculating the daily rates is:

$$
\text{D}_{i,t} = \begin{cases}
    \frac{\max(0, C_t - C_{t-1})}{H_t} \times 24 & \text{if } H_t > 0 \\
    0 & \text{if } H_t = 0
\end{cases}
$$

Where:
* $C_t$ = Cumulative oil, gas, or water production for Well $i$ on day $t$.
* $C_{t-1}$ = Cumulative oil, gas, or water production for Well $i$ on the previous recorded day ($t-1$).
* $H_t$ = `ON_STREAM_HRS` for Well $i$ on day $t$.
* $\max(0, \text{value})$ ensures that the daily production difference is not negative (clamped at 0).

Thus each rate is first expressed in **per hour** and then scaled by 24 h to obtain the standard **per day**

---


In [ ]:
# Calculate the raw daily difference in cumulative oil production
daily_oil_prod_diff = wells_df.groupby('WELL_NAME')['CUMULATIVE_OIL_PROD (STB)'].diff().fillna(0).clip(lower=0)

# Calculate daily oil production
# Handle division by zero
wells_df['DAILY_OIL_PROD (STB)'] = (daily_oil_prod_diff / wells_df['ON_STREAM_HRS']) * 24
# Replace inf values with 0, then fill any remaining NaNs with 0
wells_df['DAILY_OIL_PROD (STB)'] = wells_df['DAILY_OIL_PROD (STB)'].replace([np.inf, -np.inf], 0).fillna(0)

# Calculate the raw daily difference in cumulative formation gas production
daily_formation_gas_prod_diff = wells_df.groupby('WELL_NAME')['CUMULATIVE_FORMATION_GAS_PROD (MSCF)'].diff().fillna(0).clip(lower=0)
# Calculate daily formation gas production
wells_df['DAILY_FORMATION_GAS_PROD (MSCF)'] = (daily_formation_gas_prod_diff / wells_df['ON_STREAM_HRS']) * 24
wells_df['DAILY_FORMATION_GAS_PROD (MSCF)'] = wells_df['DAILY_FORMATION_GAS_PROD (MSCF)'].replace([np.inf, -np.inf], 0).fillna(0)


# Calculate the raw daily difference in cumulative total gas production
daily_total_gas_prod_diff = wells_df.groupby('WELL_NAME')['CUMULATIVE_TOTAL_GAS_PROD (MSCF)'].diff().fillna(0).clip(lower=0)
# Calculate daily total gas production
wells_df['DAILY_TOTAL_GAS_PROD (MSCF)'] = (daily_total_gas_prod_diff / wells_df['ON_STREAM_HRS']) * 24
wells_df['DAILY_TOTAL_GAS_PROD (MSCF)'] = wells_df['DAILY_TOTAL_GAS_PROD (MSCF)'].replace([np.inf, -np.inf], 0).fillna(0)


# Calculate the raw daily difference in cumulative water production
daily_water_prod_diff = wells_df.groupby('WELL_NAME')['CUMULATIVE_WATER_PROD (BBL)'].diff().fillna(0).clip(lower=0)
# Calculate daily water production
wells_df['DAILY_WATER_PROD (BBL)'] = (daily_water_prod_diff / wells_df['ON_STREAM_HRS']) * 24
wells_df['DAILY_WATER_PROD (BBL)'] = wells_df['DAILY_WATER_PROD (BBL)'].replace([np.inf, -np.inf], 0).fillna(0)


wells_df.describe().T

,count,mean,min,25%,50%,75%,max,std
PROD_DATE,7955,2013-08-26 16:25:49.516027648,2011-02-17 00:00:00,2012-08-02 00:00:00,2013-06-25 00:00:00,2014-10-01 00:00:00,2016-08-12 00:00:00,NaN
ON_STREAM_HRS,7955.0,21.623497,0.0,24.0,24.0,24.0,25.0,6.567826
BOTTOMHOLE_FLOWING_PRESSURE (PSI),7955.0,2613.811816,0.0,2195.0,2465.0,3067.0,4096.0,687.60064
DOWNHOLE_TEMPERATURE (deg F),7955.0,168.757429,0.0,150.2285,158.624,202.6,212.153,31.917787
ANNULUS_PRESS (PSI),7955.0,471.339542,0.0,4.199,301.247,981.329,1639.04,481.63834
CHOKE_SIZE (%),7955.0,56.155295,0.0,28.130855,51.06803,99.80095,100.0,34.975408
WELL_HEAD_PRESSURE (PSI),7955.0,212.574151,0.0,68.6615,113.718,161.1765,1787.76,282.193044
WELL_HEAD_TEMPERATURE (deg F),7955.0,86.836365,0.0,80.6775,88.364,96.7705,182.157,22.738615
CUMULATIVE_OIL_PROD (STB),7955.0,172585.46939,0.0,47691.0,109295.0,242488.5,1129301.0,174082.717158
CUMULATIVE_FORMATION_GAS_PROD (MSCF),7955.0,145623.481961,0.0,38541.5,101610.0,199103.5,1458660.0,172168.475676


In [ ]:
# Plot the daily production profiles for each well
well_names = wells_df['WELL_NAME'].unique()

for well_name in well_names:
    well_data = wells_df[wells_df['WELL_NAME'] == well_name]

    fig = go.Figure()

    # Cum. Oil Prod.
    fig.add_trace(go.Scatter(
        x=well_data['PROD_DATE'],
        y=well_data['DAILY_OIL_PROD (STB)'],
        mode='lines',
        line=dict(color='black'),
        name='Daily Oil Production',
        hovertemplate='Date: %{x}<br>Daily Oil Prod.: %{y}<extra></extra>'
    ))

    # Cum. Formation Gas Prod.
    fig.add_trace(go.Scatter(
        x=well_data['PROD_DATE'],
        y=well_data['DAILY_FORMATION_GAS_PROD (MSCF)'],
        mode='lines',
        line=dict(color='darkgrey'),
        name='Daily Formation Gas Production',
        hovertemplate='Date: %{x}<br>Daily Fm. Gas Prod.: %{y}<extra></extra>'
    ))

    # Cum. Water Prod.
    fig.add_trace(go.Scatter(
        x=well_data['PROD_DATE'],
        y=well_data['DAILY_WATER_PROD (BBL)'],
        mode='lines',
        line=dict(color='blue'),
        name='Daily Water Production',
        hovertemplate='Date: %{x}<br>Daily Water Prod.: %{y}<extra></extra>'
    ))

    fig.update_layout(
        title=f'Daily Production Profiles for {well_name}',
        xaxis_title='Date',
        yaxis_title='Daily Production',
        height=500
    )

    fig.show()


We will create a feature for daily GOR for each well $i$ on day $t$:

* Daily GOR:
$$
\text
{DAILY_GOR}_{i,t} = \frac{ \text{DAILY_FORMATION_GAS_PROD}_{i,t} \times 1000 (SCF)}{ \text{DAILY_OIL_PROD}_{i,t} (STB) }
$$

In [ ]:
# Calculate Daily GOR
# Convert MSCF to SCF
wells_df['DAILY_GOR'] = (wells_df['DAILY_FORMATION_GAS_PROD (MSCF)'] * 1000) / wells_df['DAILY_OIL_PROD (STB)']
# Handle null values
wells_df['DAILY_GOR'] = wells_df['DAILY_GOR'].replace([np.inf, -np.inf], np.nan).fillna(0)
wells_df.describe().T

,count,mean,min,25%,50%,75%,max,std
PROD_DATE,7955,2013-08-26 16:25:49.516027648,2011-02-17 00:00:00,2012-08-02 00:00:00,2013-06-25 00:00:00,2014-10-01 00:00:00,2016-08-12 00:00:00,NaN
ON_STREAM_HRS,7955.0,21.623497,0.0,24.0,24.0,24.0,25.0,6.567826
BOTTOMHOLE_FLOWING_PRESSURE (PSI),7955.0,2613.811816,0.0,2195.0,2465.0,3067.0,4096.0,687.60064
DOWNHOLE_TEMPERATURE (deg F),7955.0,168.757429,0.0,150.2285,158.624,202.6,212.153,31.917787
ANNULUS_PRESS (PSI),7955.0,471.339542,0.0,4.199,301.247,981.329,1639.04,481.63834
CHOKE_SIZE (%),7955.0,56.155295,0.0,28.130855,51.06803,99.80095,100.0,34.975408
WELL_HEAD_PRESSURE (PSI),7955.0,212.574151,0.0,68.6615,113.718,161.1765,1787.76,282.193044
WELL_HEAD_TEMPERATURE (deg F),7955.0,86.836365,0.0,80.6775,88.364,96.7705,182.157,22.738615
CUMULATIVE_OIL_PROD (STB),7955.0,172585.46939,0.0,47691.0,109295.0,242488.5,1129301.0,174082.717158
CUMULATIVE_FORMATION_GAS_PROD (MSCF),7955.0,145623.481961,0.0,38541.5,101610.0,199103.5,1458660.0,172168.475676


In [ ]:
# Calculate Daily Watercut
wells_df['DAILY_WATERCUT'] = wells_df['DAILY_WATER_PROD (BBL)'] / (wells_df['DAILY_OIL_PROD (STB)'] + wells_df['DAILY_WATER_PROD (BBL)'])
# Handle null values
wells_df['DAILY_WATERCUT'] = wells_df['DAILY_WATERCUT'].replace([np.inf, -np.inf], np.nan).fillna(0)
# Esure the Watercut cannot exceed 1
wells_df['DAILY_WATERCUT'] = wells_df['DAILY_WATERCUT'].clip(upper=1.0)
wells_df.describe().T

,count,mean,min,25%,50%,75%,max,std
PROD_DATE,7955,2013-08-26 16:25:49.516027648,2011-02-17 00:00:00,2012-08-02 00:00:00,2013-06-25 00:00:00,2014-10-01 00:00:00,2016-08-12 00:00:00,NaN
ON_STREAM_HRS,7955.0,21.623497,0.0,24.0,24.0,24.0,25.0,6.567826
BOTTOMHOLE_FLOWING_PRESSURE (PSI),7955.0,2613.811816,0.0,2195.0,2465.0,3067.0,4096.0,687.60064
DOWNHOLE_TEMPERATURE (deg F),7955.0,168.757429,0.0,150.2285,158.624,202.6,212.153,31.917787
ANNULUS_PRESS (PSI),7955.0,471.339542,0.0,4.199,301.247,981.329,1639.04,481.63834
CHOKE_SIZE (%),7955.0,56.155295,0.0,28.130855,51.06803,99.80095,100.0,34.975408
WELL_HEAD_PRESSURE (PSI),7955.0,212.574151,0.0,68.6615,113.718,161.1765,1787.76,282.193044
WELL_HEAD_TEMPERATURE (deg F),7955.0,86.836365,0.0,80.6775,88.364,96.7705,182.157,22.738615
CUMULATIVE_OIL_PROD (STB),7955.0,172585.46939,0.0,47691.0,109295.0,242488.5,1129301.0,174082.717158
CUMULATIVE_FORMATION_GAS_PROD (MSCF),7955.0,145623.481961,0.0,38541.5,101610.0,199103.5,1458660.0,172168.475676


## **Rule-Based Classification**

We will build Python codes that will classify the wells based on the parameters in `params_df`.

In [ ]:
params_df

,Reservoir Name,Reservoir Type,Well Type,Production Type,Formation GOR Trend,Watercut Trend,Oil Productivity Index Trend
0,ACHI,Saturated,NF,Steady,aSolGOR,Flat,Flat
1,KEMA,Undersat,GL,Unsteady,bSolGOR,Incr,Incr
2,MAKO,NaN,NaN,NaN,Combo,Decr,Decr
3,DEPU,NaN,NaN,NaN,NaN,Combo,Combo
4,JANI,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Create a DataFrame to store results for each well
well_classifications = pd.DataFrame(index=wells_df['WELL_NAME'].unique())
well_classifications.index.name = 'WELL_NAME'
print(well_classifications.shape)
well_classifications.head()

(20, 0)


""
WELL_NAME
Well_#1
Well_#2
Well_#3
Well_#4
Well_#5


## 1. Identify the Resevoir Name

First, we will identify the reservoir from which each well is producing. Let's look at the different reservoir types in the dataset `reservoir_df`.

In [ ]:
reservoir_df

,Reservoir Name,Initial Reservoir Pressure (PSI),Bubble Point Pressure (PSI),Current Average Reservoir Pressure (PSI),Solution Gas-Oil-Ratio (SCF/BBL),Formation Volume Factor (RB/STB)
0,ACHI,"3,500","3,300","2,700",800,1.20
1,KEMA,"4,200","4,000","3,900",600,1.45
2,MAKO,"3,500","3,500","3,000",500,1.15
3,DEPU,"2,800","2,800","2,400","1,200",1.37
4,JANI,"4,500","4,300","4,200","1,000",1.30


In [ ]:
reservoir_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Reservoir Name                            5 non-null      object 
 1   Initial Reservoir Pressure (PSI)          5 non-null      object 
 2   Bubble Point Pressure (PSI)               5 non-null      object 
 3   Current Average Reservoir Pressure (PSI)  5 non-null      object 
 4   Solution Gas-Oil-Ratio (SCF/BBL)          5 non-null      object 
 5   Formation Volume Factor (RB/STB)          5 non-null      float64
dtypes: float64(1), object(5)
memory usage: 372.0+ bytes


In [ ]:
reservoir_df.columns

Index(['Reservoir Name', 'Initial Reservoir Pressure (PSI)',
       'Bubble Point Pressure (PSI)',
       'Current Average Reservoir Pressure (PSI)',
       'Solution Gas-Oil-Ratio (SCF/BBL)', 'Formation Volume Factor (RB/STB)'],
      dtype='object')

We will need to convert the numerical columns to their actual data types.

In [ ]:
cols_to_convert = ['Initial Reservoir Pressure (PSI)',
       'Bubble Point Pressure (PSI)',
       'Current Average Reservoir Pressure (PSI)',
       'Solution Gas-Oil-Ratio (SCF/BBL)'
]
for col in cols_to_convert:
    reservoir_df[col] = reservoir_df[col].str.replace(',', '').astype(int)
reservoir_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Reservoir Name                            5 non-null      object 
 1   Initial Reservoir Pressure (PSI)          5 non-null      int64  
 2   Bubble Point Pressure (PSI)               5 non-null      int64  
 3   Current Average Reservoir Pressure (PSI)  5 non-null      int64  
 4   Solution Gas-Oil-Ratio (SCF/BBL)          5 non-null      int64  
 5   Formation Volume Factor (RB/STB)          5 non-null      float64
dtypes: float64(1), int64(4), object(1)
memory usage: 372.0+ bytes


 According to the data providers, a well is producing from a particular reservoir if there is no more than 200 psi differential pressure at the well's maximum bottom hole flowing pressure. Hence, we will compare the well's maximum bhp to the reservoir's current average reservoir pressure, since it reflects the reservoir's actual pressure state during the well's operation, including any depletion.

In [ ]:
# Get the max bottom hole flowing pressure for each well
max_bhfp_per_well = wells_df.groupby('WELL_NAME')['BOTTOMHOLE_FLOWING_PRESSURE (PSI)'].max().reset_index()
max_bhfp_per_well.columns = ['WELL_NAME', 'Max BHFP (PSI)']
# Extract the numerical part of the well name and sort by it
max_bhfp_per_well['Well_Number'] = max_bhfp_per_well['WELL_NAME'].str.extract('(\d+)').astype(int)
max_bhfp_per_well = max_bhfp_per_well.sort_values('Well_Number').drop('Well_Number', axis=1)
display(max_bhfp_per_well)

,WELL_NAME,Max BHFP (PSI)
0,Well_#1,4096.0
11,Well_#2,3735.0
13,Well_#3,2985.0
14,Well_#4,2340.0
15,Well_#5,2933.0
16,Well_#6,3797.0
17,Well_#7,3821.0
18,Well_#8,2681.0
19,Well_#9,2333.0
1,Well_#10,4051.0


In [ ]:
# Build function to identify the reservoir name
def get_reservoir_name(
        well_data: pd.DataFrame,
        reservoirs: pd.DataFrame,
        tol_psi: float = 200.0) -> str:
    """
    Decide the most plausible reservoir for a well using BottomHolePressure proximity.

    Parameters
    ----------
    well_data: All production records for one well (must include BottomHolePressure).
    reservoirs: DataFrame containing reservoir names and their current average pressures.
    tol_psi: Tolerance window (in psi) for acceptable pressure match.

    Returns
    -------
    Label string: Reservoir name when a valid match is found; otherwise "Unknown".

    Decision rules
    --------------
    1. Extract the maximum BottomHolePressure (BHP) recorded for the well.
    2. Compute the margin between each reservoir's current average pressure and the max BHP.
    3. Filter reservoirs within tol_psi of the max BHP.
    4. If no match exists within the tolerance, return "Unknown".
    5. Among matches:Prefer those with pressures above the max BHP, choose the smallest positive margin.
       If none are above, choose the one just below with the smallest absolute margin.
    """

    # Get the highest recorded BHP
    max_bhp = well_data['BOTTOMHOLE_FLOWING_PRESSURE (PSI)'].max()
    # Calculate th pressure difference
    margin = reservoirs["Current Average Reservoir Pressure (PSI)"] - max_bhp
    # Filter within tolerance
    matches = margin.abs() <= tol_psi
    if not matches.any():
        return "Unknown"

    above = margin[matches & (margin > 0)]
    if not above.empty:
        # select reservoir with smallest positive margin
        idx = above.idxmin()
        return reservoirs.at[idx, "Reservoir Name"]

    idx = margin[matches].abs().idxmin()
    return reservoirs.at[idx, "Reservoir Name"]

In [ ]:
# Iterate through each unique well name in the wells_df DataFrame
for well_name in wells_df['WELL_NAME'].unique():
    # Filter the wells_df to get data for the current well
    well_data = wells_df[wells_df['WELL_NAME'] == well_name].copy()

    # Get the reservoir name
    identified_reservoir = get_reservoir_name(well_data, reservoir_df, tol_psi=200.0)

    # Store the identified reservoir name in the well_classifications DataFrame
    well_classifications.loc[well_name, 'Reservoir Name'] = identified_reservoir

# Display the updated well_classifications DataFrame
display(well_classifications)

,Reservoir Name
WELL_NAME,
Well_#1,JANI
Well_#2,KEMA
Well_#3,MAKO
Well_#4,DEPU
Well_#5,MAKO
Well_#6,KEMA
Well_#7,KEMA
Well_#8,ACHI
Well_#9,DEPU


In [ ]:
# Count the number of wells in each reservoir
reservoir_counts = well_classifications['Reservoir Name'].value_counts().reset_index()
reservoir_counts.columns = ['Reservoir Name', 'Number of Wells']

# Create a bar chart
fig = px.bar(reservoir_counts, x='Reservoir Name', y='Number of Wells',
             title='Number of Wells per Reservoir',
             labels={'Reservoir Name': 'Reservoir', 'Number of Wells': 'Number of Wells'})
fig.update_layout(xaxis_title='Reservoir Name', yaxis_title='Number of Wells')
fig.show()

## 2. Identify the Reservoir Type

In [ ]:
reservoir_df

,Reservoir Name,Initial Reservoir Pressure (PSI),Bubble Point Pressure (PSI),Current Average Reservoir Pressure (PSI),Solution Gas-Oil-Ratio (SCF/BBL),Formation Volume Factor (RB/STB)
0,ACHI,3500,3300,2700,800,1.20
1,KEMA,4200,4000,3900,600,1.45
2,MAKO,3500,3500,3000,500,1.15
3,DEPU,2800,2800,2400,1200,1.37
4,JANI,4500,4300,4200,1000,1.30


In [ ]:
# Build a function to identify reservoir type
def get_reservoir_type(
        name: str,
        reservoirs: pd.DataFrame,
        tol_psi: float = 100.0) -> str:
    """
    Classify a reservoir as Saturated or Undersaturated based on initial pressure.

    Parameters
    ----------
    name: Reservoir name.
    reservoirs: Cleaned reservoir table containing
                'Initial Reservoir Pressure (PSI)' and
                'Bubble Point Pressure (PSI)'.
    tol_psi: Allowable cushion above bubble-point pressure still treated
             as saturated (default = 100 psi).

    Returns
    -------
    Label string: "Saturated" if within tolerance of bubble point; otherwise "Undersat".

    Decision rules
    --------------
    1. Extract the row corresponding to the given reservoir name.
    2. Check if Initial Reservoir Pressure ≤ Bubble Point Pressure + tol_psi.
    3. If yes, return "Saturated".
    4. Otherwise, return "Undersat".
    """

    # Extract matching reservoir row
    row = reservoirs.loc[reservoirs["Reservoir Name"] == name].iloc[0]

    # Determine if within saturation margin
    saturated = (
        row["Initial Reservoir Pressure (PSI)"]
        <= row["Bubble Point Pressure (PSI)"] + tol_psi
    )

    # Assign label accordingly
    return "Saturated" if saturated else "Undersat"


In [ ]:
# Determine Reservoir Type
for well_name in well_classifications.index:
    # Retrieve the identified reservoir name for this well
    identified_reservoir = well_classifications.loc[well_name, 'Reservoir Name']

    # Get reservoir type
    reservoir_type = get_reservoir_type(identified_reservoir, reservoir_df, tol_psi=100.0)

    # Store the identified reservoir type in the well_classifications DataFrame
    well_classifications.loc[well_name, 'Reservoir Type'] = reservoir_type

# Display the updated well_classifications DataFramea
display(well_classifications)

,Reservoir Name,Reservoir Type
WELL_NAME,,
Well_#1,JANI,Undersat
Well_#2,KEMA,Undersat
Well_#3,MAKO,Saturated
Well_#4,DEPU,Saturated
Well_#5,MAKO,Saturated
Well_#6,KEMA,Undersat
Well_#7,KEMA,Undersat
Well_#8,ACHI,Undersat
Well_#9,DEPU,Saturated


## 3. Identify the Well Type

We will determine whether a well is **Gas-Lifted (GL)** or **Naturally Flowing (NF)**. Three conditions must be satisfied for a well to be labeled as GL: the median annulus pressure must exceed 100 psi, at least 30% of the annulus pressure readings must exceed that threshold, and the total cumulative gas must be greater than the formation gas. If any of these conditions fail, the well is classified as NF.

In [ ]:
# Function to identify well type
def get_well_type(
        well_data: pd.DataFrame,
        annulus_threshold: float = 100.0,
        fraction_threshold: float = 0.30) -> str:
    """
    Decide whether a well is Gas-Lifted (GL) or Naturally Flowing (NF).

    Parameters
    ----------
    well_data: All production records for one well (must include annulus pressure and gas volumes).
    annulus_threshold: Pressure threshold (psi) above which annulus is considered pressurized.
    fraction_threshold: Minimum fraction of days annulus pressure must exceed threshold.

    Returns
    -------
    Label string: "GL" if all gas-lift criteria are met, otherwise "NF".

    Decision rules
    --------------
    1. Convert annulus pressure and gas volumes to numeric.
    2. Check if:
       a. Median annulus pressure > annulus_threshold.
       b. Fraction of samples above annulus_threshold ≥ fraction_threshold.
       c. Total gas produced > formation gas produced.
    3. If all conditions are met, return "GL".
    4. Otherwise, return "NF".
    """

    # Annulus pressure criteria
    ann = well_data["ANNULUS_PRESS (PSI)"]
    median_ok   = ann.median() > annulus_threshold
    fraction_ok = (ann > annulus_threshold).mean() >= fraction_threshold

    # Gas volume criterion
    total_gas     = well_data["CUMULATIVE_TOTAL_GAS_PROD (MSCF)"].max()
    formation_gas = well_data["CUMULATIVE_FORMATION_GAS_PROD (MSCF)"].max()
    gas_ok = total_gas > formation_gas

    # Final label
    return "GL" if (median_ok and fraction_ok and gas_ok) else "NF"


In [ ]:
# Determine Well Type (NF/GL)
for well_name in wells_df['WELL_NAME'].unique():
    well_data = wells_df[wells_df['WELL_NAME'] == well_name].copy()

    # Get well type
    well_type = get_well_type(well_data, annulus_threshold=100.0, fraction_threshold=0.30)

    # Store the identified well type in the well_classifications DataFrame
    well_classifications.loc[well_name, 'Well Type'] = well_type

# Display the updated well_classifications DataFrame
display(well_classifications)

,Reservoir Name,Reservoir Type,Well Type
WELL_NAME,,,
Well_#1,JANI,Undersat,GL
Well_#2,KEMA,Undersat,NF
Well_#3,MAKO,Saturated,GL
Well_#4,DEPU,Saturated,GL
Well_#5,MAKO,Saturated,NF
Well_#6,KEMA,Undersat,NF
Well_#7,KEMA,Undersat,GL
Well_#8,ACHI,Undersat,GL
Well_#9,DEPU,Saturated,GL


## 4. Identify Production Type

According to the data providers, a well has an unsteady production if it has at least 50% drop in oil production at least once every 3 to 6 months. Hence, for each well, we will label it as **Unsteady** if there is a significant drop in smoothed oil rate over time. This will be determined by computing a rolling average of daily oil rates and checking whether any future values fall below 50% of a reference point within a look-ahead window of 90 to 180 days. If no such drop is found or data is insufficient, the well will be labeled as **Steady**.

In [ ]:
# Function to identify production type
def get_production_type(
        well: pd.DataFrame,
        base_window: int = 30,
        look_ahead_min: int = 90,
        look_ahead_max: int = 180,
        drop_frac: float = 0.5) -> str:
    """
    Decide whether a well exhibits Steady or Unsteady production.

    Parameters
    ----------
    well: All production records for one well (must include Date, OnstreamHours and CumulativeOil).
    base_window: Length of the rolling mean (days) used to smooth the rate curve.
    look_ahead_min, look_ahead_max: Forward-look window (days) in which to search for a decline.
    drop_frac: Decline threshold; a drop to ≤ drop_frac of the reference rate flags Unsteady behaviour.

    Returns
    -------
    Label string: "Unsteady" when a qualifying decline is found, otherwise "Steady".

    Decision rules
    --------------
    1. Sort records by date and keep days with positive OnstreamHours.
    2. Compute daily oil-rate q = ΔOil / OnstreamHours × 24.
    3. Smooth q with a rolling mean of length base_window → q̄.
    4. At each reference date t, scan q̄ between
       [t + look_ahead_min, t + look_ahead_max].
       If any future q̄ ≤ drop_frac × q̄(t), return "Unsteady".
    5. If no such drop (or insufficient data), return "Steady".
    """

    # Filter for flowing days only
    df = well.sort_values("PROD_DATE").reset_index(drop=True).copy()
    df["ON_STREAM_HRS"] = pd.to_numeric(df["ON_STREAM_HRS"], errors="coerce")
    df = df[df["ON_STREAM_HRS"] > 0]


    if df.empty:
        return "Steady"

    # Perform rolling mean over base_window records
    df["RollRate"] = (
        df["DAILY_OIL_PROD (STB)"]
          .rolling(base_window, min_periods=base_window // 2)
          .mean()
    )

    # Perform look ahead decline test
    for _, row in df.iterrows():
        base_t, base_r = row["PROD_DATE"], row["RollRate"]
        if pd.isna(base_r):
            continue

        mask = (df["PROD_DATE"] >= base_t + timedelta(days=look_ahead_min)) & (df["PROD_DATE"] <= base_t + timedelta(days=look_ahead_max))
        future = df.loc[mask, "RollRate"]

        if (future <= drop_frac * base_r).any():
            return "Unsteady"

    return "Steady"

In [ ]:
# Determine Production Type (Steady/Unsteady)
for well_name in wells_df['WELL_NAME'].unique():
    well_data = wells_df[wells_df['WELL_NAME'] == well_name].sort_values('PROD_DATE').copy()

    # Get the production type
    production_type = get_production_type(well_data, base_window=30, look_ahead_min=90, look_ahead_max=180, drop_frac=0.5)

    # Store the identified production type in the well_classifications DataFrame
    well_classifications.loc[well_name, 'Production Type'] = production_type

# Display the updated well_classifications DataFrame
display(well_classifications)

,Reservoir Name,Reservoir Type,Well Type,Production Type
WELL_NAME,,,,
Well_#1,JANI,Undersat,GL,Unsteady
Well_#2,KEMA,Undersat,NF,Steady
Well_#3,MAKO,Saturated,GL,Steady
Well_#4,DEPU,Saturated,GL,Steady
Well_#5,MAKO,Saturated,NF,Unsteady
Well_#6,KEMA,Undersat,NF,Unsteady
Well_#7,KEMA,Undersat,GL,Unsteady
Well_#8,ACHI,Undersat,GL,Steady
Well_#9,DEPU,Saturated,GL,Unsteady


In [ ]:
# Plot the daily oil production profiles for each well
well_names = wells_df['WELL_NAME'].unique()
# Define the desired rolling window size
rolling_window_size = 30

for well_name in well_names:
    well_data = wells_df[wells_df['WELL_NAME'] == well_name].sort_values('PROD_DATE').copy()

    # Calculate the rolling average of daily oil production
    well_data['SMOOTHED_DAILY_OIL_PROD'] = well_data['DAILY_OIL_PROD (STB)'].rolling(window=rolling_window_size, min_periods=1).mean()

    fig = go.Figure()

    # Daily Oil Prod. trace
    fig.add_trace(go.Scatter(
        x=well_data['PROD_DATE'],
        y=well_data['DAILY_OIL_PROD (STB)'],
        mode='lines',
        line=dict(color='black'),
        opacity=0.5,
        name='Daily Oil Production',
        hovertemplate='Date: %{x}<br>Daily Oil Prod.: %{y}<extra></extra>'
    ))

    # Smoothed Daily Oil Prod. trace
    fig.add_trace(go.Scatter(
        x=well_data['PROD_DATE'],
        y=well_data['SMOOTHED_DAILY_OIL_PROD'],
        mode='lines',
        line=dict(color='red', width=2),
        name=f'Smoothed Daily Oil Prod. ({rolling_window_size}-day Avg)',
        hovertemplate='Date: %{x}<br>Smoothed Oil Prod.: %{y:.2f}<extra></extra>'
    ))


    fig.update_layout(
        title=f'Daily and Smoothed Oil Production Profile for {well_name}',
        xaxis_title='Date',
        yaxis_title='Daily Oil Production (STB)',
        height=500
    )

    fig.show()

## 5. Identify the Formation GOR Trend

To identify the Formation GOR Trend for each well, we will analyze the monthly gas-oil ratio (GOR) data for each well and compare it to the corresponding reservoir’s solution GOR to determine if the trend is dominantly above (`aSolGOR`), below (`bSolGOR`), or mixed (`Combo`). We will use a default threshold of 80% to decide dominance.


In [ ]:
def get_formation_gor_trend(
        well_data: pd.DataFrame,
        reservoir_df: pd.DataFrame,
        well_classifications: pd.DataFrame,
        trend_threshold: float = 0.8) -> str:
    """
    Classify the Formation GOR Trend for a well as 'aSolGOR', 'bSolGOR', or 'Combo'
    using monthly GOR calculation and comparison to solution GOR.

    Parameters
    ----------
    well_data: DataFrame for one well (must include 'PROD_DATE', 'WELL_NAME',
               'CUMULATIVE_OIL_PROD (STB)', 'CUMULATIVE_FORMATION_GAS_PROD (MSCF)').
    reservoir_df: DataFrame containing reservoir names and their Solution Gas-Oil-Ratio (SCF/BBL).
    well_classifications: DataFrame with 'WELL_NAME' as index and 'Reservoir Name' column.
    trend_threshold: Proportion (0–1) of GOR values that must be dominant to assign a trend label.

    Returns
    -------
    Label string: 'aSolGOR', 'bSolGOR', 'Combo', 'Unknown Reservoir', or 'Insufficient Data'.
    """

    # Get the well name
    well_name = well_data['WELL_NAME'].iloc[0]

    # Get the associated reservoir name
    if well_name in well_classifications.index:
        identified_reservoir = well_classifications.loc[well_name, 'Reservoir Name']
    else:
        return 'Unknown Reservoir'

    # Validate reservoir existence
    if identified_reservoir not in reservoir_df['Reservoir Name'].values:
        return 'Unknown Reservoir'

    # Get the solution GOR
    solution_gor = reservoir_df[
        reservoir_df['Reservoir Name'] == identified_reservoir
    ]['Solution Gas-Oil-Ratio (SCF/BBL)'].iloc[0]

    # Monthly aggregation and GOR calculation
    current_well_data = well_data.copy()
    current_well_data['Month'] = current_well_data['PROD_DATE'].dt.to_period('M')
    monthly = current_well_data.groupby('Month').last().reset_index()

    monthly['GAS_SCF_DIFF'] = monthly['CUMULATIVE_FORMATION_GAS_PROD (MSCF)'].diff().fillna(0)
    monthly['OIL_STB_DIFF'] = monthly['CUMULATIVE_OIL_PROD (STB)'].diff().fillna(0)
    monthly['GOR'] = (monthly['GAS_SCF_DIFF'] * 1000) / monthly['OIL_STB_DIFF']
    monthly['GOR'] = monthly['GOR'].replace([np.inf, -np.inf], np.nan)

    # Filter out NaN and zero GOR values for trend analysis
    gors = monthly['GOR'].dropna()
    gors = gors[gors != 0] # Exclude zero GOR values

    if gors.empty:
        return 'Insufficient Data'

    # Count dominance
    above = (gors > solution_gor).sum()
    below_or_equal = (gors <= solution_gor).sum()
    total = above + below_or_equal

    if total == 0:
        return 'Insufficient Data'

    # Apply threshold
    if above / total >= trend_threshold:
        return 'aSolGOR'
    elif below_or_equal / total >= trend_threshold:
        return 'bSolGOR'
    else:
        return 'Combo'

In [ ]:
for well_name in wells_df['WELL_NAME'].unique():
    well_data = wells_df[wells_df['WELL_NAME'] == well_name].sort_values('PROD_DATE').copy()

    # Get the Formation GOR Trend
    formation_gor_trend = get_formation_gor_trend(
        well_data,
        reservoir_df,
        well_classifications,
        trend_threshold=0.8
    )

    # Store the identified Formation GOR Trend in the well_classifications DataFrame
    well_classifications.loc[well_name, 'Formation GOR Trend'] = formation_gor_trend

display(well_classifications)

,Reservoir Name,Reservoir Type,Well Type,Production Type,Formation GOR Trend
WELL_NAME,,,,,
Well_#1,JANI,Undersat,GL,Unsteady,aSolGOR
Well_#2,KEMA,Undersat,NF,Steady,Combo
Well_#3,MAKO,Saturated,GL,Steady,aSolGOR
Well_#4,DEPU,Saturated,GL,Steady,aSolGOR
Well_#5,MAKO,Saturated,NF,Unsteady,aSolGOR
Well_#6,KEMA,Undersat,NF,Unsteady,aSolGOR
Well_#7,KEMA,Undersat,GL,Unsteady,aSolGOR
Well_#8,ACHI,Undersat,GL,Steady,aSolGOR
Well_#9,DEPU,Saturated,GL,Unsteady,bSolGOR


In [ ]:
# Get unique well names
well_names = wells_df['WELL_NAME'].unique()

for well_name in well_names:
    # Filter data for the current well
    well_data = wells_df[wells_df['WELL_NAME'] == well_name].sort_values('PROD_DATE').copy()

    # Filter out zero DAILY_GOR values for plotting
    well_data_filtered = well_data[well_data['DAILY_GOR'] != 0].copy()


    # Create a Plotly figure
    fig = go.Figure()

    # Add Daily GOR trace
    fig.add_trace(go.Scatter(
        x=well_data_filtered['PROD_DATE'],
        y=well_data_filtered['DAILY_GOR'],
        mode='lines',
        line=dict(color='blue'),
        name='Daily GOR (excluding zeros)',
        hovertemplate='Date: %{x}<br>Daily GOR: %{y:.2f}<extra></extra>'
    ))


    # Update layout
    fig.update_layout(
        title=f'Daily GOR Profile for {well_name} (excluding zeros)', # Updated title
        xaxis_title='Date',
        yaxis_title='GOR (SCF/BBL)',
        height=500
    )

    # Show the plot
    fig.show()

## 6. Identify Watercut Trend



The watercut trend is derived from the ratio of daily water production to the sum of daily oil and water production. We will identify this trend by using a rule-based function that smooths the watercut data and checks the dominant direction of change. Depending on the pattern, the trend will be labeled as 'Flat', 'Incr' (increasing), 'Decr' (decreasing), or 'Combo'.



In [ ]:
# Helper function to classify water cut trend
def _classify_wc_trend(series: np.ndarray, window: int = 19, flat_tol: float = 0.09999999999999999, eps: float = 0.001) -> str:
    """
    Classify the trend in a time series as Flat, Increasing, Decreasing, or Combo.

    Parameters
    ----------
    series: Time series array to classify.
    window: Size of the rolling window for median smoothing.
    flat_tol: Tolerance for how flat a signal must be to count as “Flat”.
    eps: Minimum change in value to count as a meaningful slope.

    Returns
    -------
    Label string: One of "Flat", "Incr", "Decr", or "Combo".

    Decision rules
    --------------
    1. If fewer than 3 points, return "Flat".
    2. Apply median smoothing using the specified window.
    3. Compute deltas between smoothed points.
    4. Calculate fraction of positive and negative slopes.
    5. If both fractions ≤ flat_tol, return "Flat".
    6. If only negative fraction ≤ flat_tol, return "Incr".
    7. If only positive fraction ≤ flat_tol, return "Decr".
    8. Otherwise, return "Combo".
    """

    if len(series) < 3:
        return "Flat"

    smooth = pd.Series(series).rolling(window, center=True, min_periods=1).median().values
    deltas = np.diff(smooth)

    # Define fraction of positive slopes
    pos_frac = (deltas > eps).mean()
    # Define fraction of negative slopes
    neg_frac = (deltas < -eps).mean()

    if pos_frac <= flat_tol and neg_frac <= flat_tol:
        return "Flat"
    if neg_frac <= flat_tol:
        return "Incr"
    if pos_frac <= flat_tol:
        return "Decr"
    return "Combo"


# Main function to classify water cut trend
def get_watercut_trend(well: pd.DataFrame) -> str:
    """
    Classify the trend in water cut over time using the _classify_trend method.

    Parameters
    ----------
    well: All production records for one well (must include PROD_DATE, CUMULATIVE_OIL_PROD (STB), and CUMULATIVE_WATER_PROD (BBL)).

    Returns
    -------
    Label string: One of "Flat", "Incr", "Decr", or "Combo".

    Decision rules
    --------------
    1. Sort records by date.
    2. Compute ΔOil and ΔWater for each day, clip negatives.
    3. Filter out rows where both deltas are zero.
    4. Compute daily water cut = ΔWater / (ΔOil + ΔWater).
    5. Classify the resulting series using _classify_trend.
    """

    df = well.sort_values("PROD_DATE").assign(
        OilDiff=lambda d: d["CUMULATIVE_OIL_PROD (STB)"].diff().clip(lower=0),
        WaterDiff=lambda d: d["CUMULATIVE_WATER_PROD (BBL)"].diff().clip(lower=0))

    # Filter for non-zero production values
    df = df[(df["OilDiff"] + df["WaterDiff"]) > 0]

    # Handle potential division by zero if OilDiff + WaterDiff is 0
    valid_watercut_mask = (df["OilDiff"] + df["WaterDiff"]) > 0
    wc = pd.Series(np.nan, index=df.index) #
    wc[valid_watercut_mask] = df.loc[valid_watercut_mask, "WaterDiff"] / (df.loc[valid_watercut_mask, "OilDiff"] + df.loc[valid_watercut_mask, "WaterDiff"])

    # Handle potential NaNs introduced by division by zero or initial diff calculation
    wc = wc.fillna(0)

    return _classify_wc_trend(wc.values)

In [ ]:
for well_name in wells_df['WELL_NAME'].unique():
    well_data = wells_df[wells_df['WELL_NAME'] == well_name].sort_values('PROD_DATE').copy()

    # Get the Water cut Trend
    watercut_trend = get_watercut_trend(well_data)

    # Store the identified Water Cut Trend in the well_classifications DataFrame
    well_classifications.loc[well_name, 'Watercut Trend'] = watercut_trend

display(well_classifications)

,Reservoir Name,Reservoir Type,Well Type,Production Type,Formation GOR Trend,Watercut Trend
WELL_NAME,,,,,,
Well_#1,JANI,Undersat,GL,Unsteady,aSolGOR,Incr
Well_#2,KEMA,Undersat,NF,Steady,Combo,Combo
Well_#3,MAKO,Saturated,GL,Steady,aSolGOR,Flat
Well_#4,DEPU,Saturated,GL,Steady,aSolGOR,Combo
Well_#5,MAKO,Saturated,NF,Unsteady,aSolGOR,Incr
Well_#6,KEMA,Undersat,NF,Unsteady,aSolGOR,Flat
Well_#7,KEMA,Undersat,GL,Unsteady,aSolGOR,Incr
Well_#8,ACHI,Undersat,GL,Steady,aSolGOR,Incr
Well_#9,DEPU,Saturated,GL,Unsteady,bSolGOR,Incr


In [ ]:
import plotly.graph_objects as go

# Get unique well names
well_names = wells_df['WELL_NAME'].unique()

for well_name in well_names:
    # Filter data for the current well
    well_data = wells_df[wells_df['WELL_NAME'] == well_name].sort_values('PROD_DATE').copy()

    # Filter out zero DAILY_WATERCUT values for plotting
    well_data_filtered = well_data[well_data['DAILY_WATERCUT'] != 0].copy()

    # Create a Plotly figure
    fig = go.Figure()

    # Add Daily Watercut trace (using filtered data)
    fig.add_trace(go.Scatter(
        x=well_data_filtered['PROD_DATE'], # Use filtered data for x-axis
        y=well_data_filtered['DAILY_WATERCUT'], # Use filtered data for y-axis
        mode='lines',
        line=dict(color='blue'),
        name='Daily Watercut (excluding zeros)', # Updated name to reflect filtering
        hovertemplate='Date: %{x}<br>Daily Watercut: %{y:.4f}<extra></extra>'
    ))

    # Update layout
    fig.update_layout(
        title=f'Daily Watercut Profile for {well_name} (excluding zeros)', # Updated title
        xaxis_title='Date',
        yaxis_title='Watercut (Ratio)',
        height=500
    )

    # Show the plot
    fig.show()

## 7. Identify oil productivity index trend

The OPI is calculated as the ratio of oil rate to drawdown (difference between average reservoir pressure and BHP). We will analyze the OPI trend using the same rule-based trend classifier as watercut. Based on how the OPI changes, the well is labeled as 'Flat', 'Incr', 'Decr', or 'Combo'.

In [ ]:
# Helper function to classify OPI trend
def _classify_opi_trend(series: np.ndarray, window: int = 3, flat_tol: float = 0.044, eps: float = 0.064) -> str:
    """
    Classify the trend in a time series as Flat, Increasing, Decreasing, or Combo.

    Parameters
    ----------
    series: Time series array to classify.
    window: Size of the rolling window for median smoothing.
    flat_tol: Tolerance for how flat a signal must be to count as “Flat”.
    eps: Minimum change in value to count as a meaningful slope.

    Returns
    -------
    Label string: One of "Flat", "Incr", "Decr", or "Combo".

    Decision rules
    --------------
    1. If fewer than 3 points, return "Flat".
    2. Apply median smoothing using the specified window.
    3. Compute deltas between smoothed points.
    4. Calculate fraction of positive and negative slopes.
    5. If both fractions ≤ flat_tol, return "Flat".
    6. If only negative fraction ≤ flat_tol, return "Incr".
    7. If only positive fraction ≤ flat_tol, return "Decr".
    8. Otherwise, return "Combo".
    """

    if len(series) < 3:
        return "Flat"

    smooth = pd.Series(series).rolling(window, center=True, min_periods=1).median().values
    deltas = np.diff(smooth)

    # Define fraction of positive slopes
    pos_frac = (deltas > eps).mean()
    # Define fraction of negative slope
    neg_frac = (deltas < -eps).mean()

    if pos_frac <= flat_tol and neg_frac <= flat_tol:
        return "Flat"
    if neg_frac <= flat_tol:
        return "Incr"
    if pos_frac <= flat_tol:
        return "Decr"
    return "Combo"


def get_opi_trend(
        well_data: pd.DataFrame,
        reservoirs: pd.DataFrame,
        reservoir_name: str) -> str:
    """
    Classify the Oil-rate Productivity Index (OPI) trend for a well.

    Parameters
    ----------
    well_data: All production records for one well (must include PROD_DATE, ON_STREAM_HRS,
               CUMULATIVE_OIL_PROD (STB), and BOTTOMHOLE_FLOWING_PRESSURE (PSI)).
    reservoirs: Master table of reservoir properties (must include “Reservoir Name”
               and “Current Average Reservoir Pressure (PSI)”).
    reservoir_name: Reservoir name already associated with the well. # Updated parameter description

    Returns
    -------
    Trend label string produced by _classify_trend(), e.g. "Flat", "Incr", "Decr", "Combo".

    Decision rules
    --------------
    1. Look up the reservoir’s current average pressure (avg_pres).
    2. Sort the well’s records chronologically and keep flowing days (OnstreamHours > 0).
    3. Compute daily oil-rate   q = ΔOil / OnstreamHours × 24.
    4. Compute drawdown         Δp = avg_pres – BottomHolePressure.
    5. OPI vector               OPI = q / Δp  (units: STB per psi).
    6. Run _classify_trend(OPI) to label the trajectory.
    """


    # Get the average pressure according to reservoir name
    avg_pres = reservoirs.loc[
        reservoirs["Reservoir Name"] == reservoir_name,
        "Current Average Reservoir Pressure (PSI)"
    ].iloc[0]

    # Filter for flowing days only
    df = well_data.sort_values("PROD_DATE").reset_index(drop=True).copy()
    df["ON_STREAM_HRS"] = pd.to_numeric(df["ON_STREAM_HRS"], errors="coerce")
    df = df[df["ON_STREAM_HRS"] > 0].copy() # Add .copy() to avoid SettingWithCopyWarning

    if df.empty:
        return "Flat"


    # Calulate the daily oil rate
    df["OilDiff"] = df["CUMULATIVE_OIL_PROD (STB)"].diff().clip(lower=0)
    df = df[df["OilDiff"] > 0].copy()
    # Handle potential division by zero if ON_STREAM_HRS is 0
    valid_rate_mask = df["ON_STREAM_HRS"] > 0
    df.loc[valid_rate_mask, "OilRate"] = (df.loc[valid_rate_mask, "OilDiff"] / df.loc[valid_rate_mask, "ON_STREAM_HRS"]) * 24
    # Handle inf/NaN and fill with 0
    df['OilRate'] = df['OilRate'].replace([np.inf, -np.inf], np.nan).fillna(0)


    # Calculate the pressure drawdown
    df["Drawdown"] = avg_pres - df["BOTTOMHOLE_FLOWING_PRESSURE (PSI)"]

    # Filter for positive drawdown as PI is typically defined for positive drawdown
    df = df[(df["OilRate"] > 0) & (df["Drawdown"] > 0)].copy()

    if df.empty:
         return "Flat"

    # Calculate OPI
    opi = df["OilRate"] / df["Drawdown"]

    # Use the _classify_opi_trend helper function for classification
    return _classify_opi_trend(opi.values)

In [ ]:
for well_name in wells_df['WELL_NAME'].unique():
    well_data = wells_df[wells_df['WELL_NAME'] == well_name].sort_values('PROD_DATE').copy()

    # Get the identified reservoir name for this well from well_classifications
    identified_reservoir = well_classifications.loc[well_name, 'Reservoir Name']

    # Get the OPI trend - Pass the identified_reservoir name
    opi_trend = get_opi_trend(well_data, reservoir_df, identified_reservoir)

    # Store the identified OPI Trend in the well_classifications DataFrame
    well_classifications.loc[well_name, 'Oil Productivity Index'] = opi_trend

display(well_classifications)

,Reservoir Name,Reservoir Type,Well Type,Production Type,Formation GOR Trend,Watercut Trend,Oil Productivity Index
WELL_NAME,,,,,,,
Well_#1,JANI,Undersat,GL,Unsteady,aSolGOR,Incr,Decr
Well_#2,KEMA,Undersat,NF,Steady,Combo,Combo,Combo
Well_#3,MAKO,Saturated,GL,Steady,aSolGOR,Flat,Combo
Well_#4,DEPU,Saturated,GL,Steady,aSolGOR,Combo,Combo
Well_#5,MAKO,Saturated,NF,Unsteady,aSolGOR,Incr,Flat
Well_#6,KEMA,Undersat,NF,Unsteady,aSolGOR,Flat,Combo
Well_#7,KEMA,Undersat,GL,Unsteady,aSolGOR,Incr,Combo
Well_#8,ACHI,Undersat,GL,Steady,aSolGOR,Incr,Combo
Well_#9,DEPU,Saturated,GL,Unsteady,bSolGOR,Incr,Combo


In [ ]:
# Calculate Daily Productivity Index (PI) and plot for each well

well_names = wells_df['WELL_NAME'].unique()

for well_name in well_names:
    well_data = wells_df[wells_df['WELL_NAME'] == well_name].sort_values('PROD_DATE').copy()

    # Retrieve the current average reservoir pressure for the well's reservoir
    identified_reservoir = well_classifications.loc[well_name, 'Reservoir Name']
    # Handle the case where the reservoir might be 'Unknown' or not found in reservoir_df
    if identified_reservoir in reservoir_df['Reservoir Name'].values:
        avg_reservoir_pressure = reservoir_df[reservoir_df['Reservoir Name'] == identified_reservoir]['Current Average Reservoir Pressure (PSI)'].iloc[0]
    else:
        avg_reservoir_pressure = np.nan


    # Calculate Daily Productivity Index (PI)
    # Handle potential division by zero or negative pressure differentials
    if pd.notnull(avg_reservoir_pressure):
        pressure_differential = avg_reservoir_pressure - well_data['BOTTOMHOLE_FLOWING_PRESSURE (PSI)']
        # Avoid division by zero or non-positive differentials where PI is undefined or non-physical
        valid_pi_mask = pressure_differential > 0
        well_data.loc[valid_pi_mask, 'DAILY_PI'] = well_data.loc[valid_pi_mask, 'DAILY_OIL_PROD (STB)'] / pressure_differential[valid_pi_mask]
        well_data.loc[~valid_pi_mask, 'DAILY_PI'] = np.nan
    else:
         well_data['DAILY_PI'] = np.nan

    # Fill potential NaNs in DAILY_PI with 0 after calculation
    well_data['DAILY_PI'] = well_data['DAILY_PI'].replace([np.inf, -np.inf], np.nan).fillna(0)


    # Plot the daily PI and smoothed PI
    fig = go.Figure()

    # Add Daily PI trace
    fig.add_trace(go.Scatter(
        x=well_data['PROD_DATE'],
        y=well_data['DAILY_PI'],
        mode='lines',
        line=dict(color='green'),
        opacity=0.5, # Corrected: opacity is a direct argument to go.Scatter
        name='Daily PI',
        hovertemplate='Date: %{x}<br>Daily PI: %{y:.4f}<extra></extra>'
    ))

    fig.update_layout(
        title=f'Daily Productivity Index Profile for {well_name}',
        xaxis_title='Date',
        yaxis_title='Productivity Index',
        height=500
    )

    fig.show()

## Total Reservoir Barrels of Oil (STB)

In [ ]:
# Get the latest cumulative oil production for each well from wells_df
latest_cumulative_prod = wells_df.groupby('WELL_NAME')['CUMULATIVE_OIL_PROD (STB)'].max().reset_index()

# Reset the index of well_classifications to make 'WELL_NAME' a column
well_classifications_reset = well_classifications.reset_index()

# Merge latest_cumulative_prod  with well_classifications DataFrame
merged_prod_classification = pd.merge(
    latest_cumulative_prod,
    well_classifications_reset[['WELL_NAME', 'Reservoir Name']], # Use the DataFrame with reset index
    on='WELL_NAME'
)

# Group by Reservoir Name and sum the latest cumulative oil production
total_oil_by_reservoir = merged_prod_classification.groupby('Reservoir Name')['CUMULATIVE_OIL_PROD (STB)'].sum().reset_index()

# Rename the cumulative production column for clarity
total_oil_by_reservoir = total_oil_by_reservoir.rename(columns={'CUMULATIVE_OIL_PROD (STB)': 'Total Surface Oil (STB)'})

# Merge with reservoir_df to get the Formation Volume Factor
total_oil_by_reservoir = pd.merge(
    total_oil_by_reservoir,
    reservoir_df[['Reservoir Name', 'Formation Volume Factor (RB/STB)']],
    on='Reservoir Name'
)

# Calculate Total Reservoir Oil (RB)
total_oil_by_reservoir['Total Reservoir Oil (RB)'] = total_oil_by_reservoir['Total Surface Oil (STB)'] * total_oil_by_reservoir['Formation Volume Factor (RB/STB)']

total_oil_by_reservoir

,Reservoir Name,Total Surface Oil (STB),Formation Volume Factor (RB/STB),Total Reservoir Oil (RB)
0,ACHI,1250123.0,1.20,1500147.60
1,DEPU,1104470.0,1.37,1513123.90
2,JANI,457241.0,1.30,594413.30
3,KEMA,1860889.0,1.45,2698289.05
4,MAKO,966013.0,1.15,1110914.95


In [ ]:
# Plot the total reservoir barrels
fig = px.bar(total_oil_by_reservoir, x='Reservoir Name', y='Total Reservoir Oil (RB)',
             title='Total Reservoir Barrels of Oil (RB) per Reservoir',
             labels={'Reservoir Name': 'Reservoir', 'Total Reservoir Oil (RB)': 'Total Oil (RB)'})
fig.update_layout(xaxis_title='Reservoir Name', yaxis_title='Total Oil (RB)')
fig.show()

## **Machine Learning Model**

Next, we are going to build a multi-output Convolutional Neural Network (CNN) to simultaneously predict the multiple well characteristics.

In [ ]:
# Ensure reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Define the sequence length for our model (the number of consecutive data points or time steps that will be used as input for the model for each sample)
SEQ_LEN = 60

# Define feature and label columns
FEATURE_COLS = ['ON_STREAM_HRS',
       'BOTTOMHOLE_FLOWING_PRESSURE (PSI)', 'DOWNHOLE_TEMPERATURE (deg F)',
       'ANNULUS_PRESS (PSI)', 'CHOKE_SIZE (%)', 'WELL_HEAD_PRESSURE (PSI)',
       'WELL_HEAD_TEMPERATURE (deg F)', 'CUMULATIVE_OIL_PROD (STB)',
       'CUMULATIVE_FORMATION_GAS_PROD (MSCF)',
       'CUMULATIVE_TOTAL_GAS_PROD (MSCF)', 'CUMULATIVE_WATER_PROD (BBL)',
       'DAILY_OIL_PROD (STB)', 'DAILY_FORMATION_GAS_PROD (MSCF)',
       'DAILY_TOTAL_GAS_PROD (MSCF)', 'DAILY_WATER_PROD (BBL)', 'DAILY_GOR',
       'DAILY_WATERCUT'
]

LABEL_COLS = [
    "Reservoir Name", "Reservoir Type", "Well Type",
    "Production Type", "Formation GOR Trend", "Watercut Trend", "Oil Productivity Index"
]

In [ ]:
# Merge the datasets on the well name
data = wells_df.merge(well_classifications, on="WELL_NAME", how="inner")
data.head()

,PROD_DATE,WELL_NAME,ON_STREAM_HRS,BOTTOMHOLE_FLOWING_PRESSURE (PSI),DOWNHOLE_TEMPERATURE (deg F),ANNULUS_PRESS (PSI),CHOKE_SIZE (%),WELL_HEAD_PRESSURE (PSI),WELL_HEAD_TEMPERATURE (deg F),CUMULATIVE_OIL_PROD (STB),...,DAILY_WATER_PROD (BBL),DAILY_GOR,DAILY_WATERCUT,Reservoir Name,Reservoir Type,Well Type,Production Type,Formation GOR Trend,Watercut Trend,Oil Productivity Index
0,2014-02-15,Well_#1,0.0,4050.0,189.866,0.0,1.17951,482.460,50.864,0.0,...,0.0,0.0,0.0,JANI,Undersat,GL,Unsteady,aSolGOR,Incr,Decr
1,2014-02-16,Well_#1,0.0,3961.0,189.945,0.0,2.99440,328.601,47.668,0.0,...,0.0,0.0,0.0,JANI,Undersat,GL,Unsteady,aSolGOR,Incr,Decr
2,2014-02-17,Well_#1,0.0,3961.0,190.004,0.0,1.90349,387.218,48.962,0.0,...,0.0,0.0,0.0,JANI,Undersat,GL,Unsteady,aSolGOR,Incr,Decr
3,2014-02-18,Well_#1,0.0,3964.0,190.020,0.0,0.00000,308.980,46.636,0.0,...,0.0,0.0,0.0,JANI,Undersat,GL,Unsteady,aSolGOR,Incr,Decr
4,2014-02-19,Well_#1,0.0,3965.0,190.107,0.0,30.20760,196.057,47.297,0.0,...,0.0,0.0,0.0,JANI,Undersat,GL,Unsteady,aSolGOR,Incr,Decr


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7955 entries, 0 to 7954
Data columns (total 26 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   PROD_DATE                             7955 non-null   datetime64[ns]
 1   WELL_NAME                             7955 non-null   object        
 2   ON_STREAM_HRS                         7955 non-null   float64       
 3   BOTTOMHOLE_FLOWING_PRESSURE (PSI)     7955 non-null   float64       
 4   DOWNHOLE_TEMPERATURE (deg F)          7955 non-null   float64       
 5   ANNULUS_PRESS (PSI)                   7955 non-null   float64       
 6   CHOKE_SIZE (%)                        7955 non-null   float64       
 7   WELL_HEAD_PRESSURE (PSI)              7955 non-null   float64       
 8   WELL_HEAD_TEMPERATURE (deg F)         7955 non-null   float64       
 9   CUMULATIVE_OIL_PROD (STB)             7955 non-null   float64       
 10  

We are going to prepare the data for our model by creating input features and corresponding labels. We will iterate through each well's production data, extract a fixed-length sequence of features (60), and pad shorter sequences with zeros. We will also encode the categorical labels into numerical format using `LabelEncoder`. Finally, we will compile these sequences and encoded labels into NumPy arrays, X and y, ready for use in training our time-series classification model.

In [ ]:
# Build model input (X) and labels (y)

# Define and fit encoders for each label column
encoders = {}
for col in LABEL_COLS:
    encoders[col] = LabelEncoder()
    encoders[col].fit(well_classifications[col])

X_list, y_list = [], []
for well, grp in data.groupby("WELL_NAME"):
    # Extract first SEQ_LEN timesteps
    seq = grp[FEATURE_COLS].values[:SEQ_LEN]
    if seq.shape[0] < SEQ_LEN:
        # Pad with zeros
        seq = np.vstack([seq, np.zeros((SEQ_LEN - seq.shape[0], len(FEATURE_COLS)))])
    X_list.append(seq)
    # Get label row for this well
    row = well_classifications[well_classifications.index == well].iloc[0]
    # Encode each label
    y_list.append([encoders[c].transform([row[c]])[0] for c in LABEL_COLS])

X = np.array(X_list)
y = np.array(y_list)
print(X[0:1])
print("")
print(y)

[[[0.00000000e+00 4.05000000e+03 1.89866000e+02 ... 0.00000000e+00
   0.00000000e+00 0.00000000e+00]
  [0.00000000e+00 3.96100000e+03 1.89945000e+02 ... 0.00000000e+00
   0.00000000e+00 0.00000000e+00]
  [0.00000000e+00 3.96100000e+03 1.90004000e+02 ... 0.00000000e+00
   0.00000000e+00 0.00000000e+00]
  ...
  [2.40000000e+01 2.87800000e+03 2.08292000e+02 ... 9.10000000e+01
   1.01884058e+03 1.16517286e-01]
  [2.40000000e+01 2.88200000e+03 2.08273000e+02 ... 8.50000000e+01
   1.03554869e+03 1.16120219e-01]
  [2.40000000e+01 2.85000000e+03 2.08396000e+02 ... 9.40000000e+01
   1.02546917e+03 1.11904762e-01]]]

[[2 1 0 1 1 2 1]
 [2 1 1 0 1 1 0]
 [0 1 0 0 1 2 0]
 [0 1 1 1 2 2 0]
 [1 0 1 0 1 1 0]
 [4 0 1 0 2 1 0]
 [2 1 1 0 0 2 1]
 [3 1 1 1 1 1 2]
 [1 0 0 0 2 0 0]
 [0 1 1 0 1 2 0]
 [2 1 1 0 0 2 2]
 [3 1 1 0 0 0 0]
 [4 0 0 0 0 2 1]
 [4 0 0 0 1 1 0]
 [1 0 0 0 1 0 0]
 [4 0 1 1 1 2 2]
 [3 1 1 1 1 1 0]
 [3 1 0 1 1 2 0]
 [0 1 0 0 1 2 0]
 [1 0 0 1 2 2 0]]


In [ ]:
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (20, 60, 17)
y shape: (20, 7)


Here, the first dimension, 20, represents the number of samples, which corresponds to the 20 unique wells in our dataset. The second dimension, 60, represents the sequence length (SEQ_LEN). This means each sample (each well) is represented by a sequence of 60 consecutive time steps or data points from its production history. The third dimension, 17, represents the number of features being used for each time step.

In [ ]:
# Standardize feature values -
n_wells, _, n_feat = X.shape

scaler = StandardScaler()
# Flatten X and scale
X_flat = scaler.fit_transform(X.reshape(-1, n_feat))
# Reshape back to original format
X = X_flat.reshape(n_wells, SEQ_LEN, n_feat)
X[0]

array([[-2.98268421,  2.01936337,  0.65519584, ..., -0.75537299,
        -2.0642835 , -1.17542593],
       [-2.98268421,  1.89164782,  0.65729544, ..., -0.75537299,
        -2.0642835 , -1.17542593],
       [-2.98268421,  1.89164782,  0.65886348, ..., -0.75537299,
        -2.0642835 , -1.17542593],
       ...,
       [ 0.40139531,  0.33753617,  1.14490568, ..., -0.5690441 ,
         0.42438373, -0.55897399],
       [ 0.40139531,  0.34327619,  1.14440071, ..., -0.58132952,
         0.46519572, -0.56107473],
       [ 0.40139531,  0.29735599,  1.1476697 , ..., -0.56290139,
         0.44057503, -0.58337723]])

In [ ]:
# Convert label encodings to One-Hot representations
num_classes = {}
y_oh = {}
for i, col in enumerate(LABEL_COLS):
    cls = encoders[col].classes_
    num = len(cls)
    num_classes[col] = num
    y_oh[col] = to_categorical(y[:, i], num_classes=num)

In [ ]:
# print the one-hot encoded array for the first element, the 'Reservoir Name'
y_oh[LABEL_COLS[0]]

array([[0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.]])

We will define the layers of our **convolutional neural network (CNN)**. It will involve sequential convolution, pooling, and dense layers to extract temporal features from the input sequences.

In [ ]:
# Define the multi-output CNN model
inp = Input(shape=(SEQ_LEN, n_feat))

x = layers.Conv1D(64, 3, activation="relu", padding="same")(inp)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(128, 3, activation="relu", padding="same")(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)

outputs = []
for col in LABEL_COLS:
    # Create a valid layer name by replacing spaces with underscores
    layer_name = col.replace(' ', '_')
    outputs.append(layers.Dense(num_classes[col], activation="softmax", name=layer_name)(x))


model = models.Model(inp, outputs)

model.compile(
    optimizer="adam",
    loss={col.replace(' ', '_'): "categorical_crossentropy" for col in LABEL_COLS},
    metrics={col.replace(' ', '_'): "accuracy" for col in LABEL_COLS}
)

In [ ]:
# Print model architecture
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 60, 17)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 60, 64)    │      3,328 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 30, 64)    │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 30, 128)   │     24,704 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     16,512 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Reservoir_Name      │ (None, 5)         │        645 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Reservoir_Type      │ (None, 2)         │        258 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Well_Type (Dense)   │ (None, 2)         │        258 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Production_Type     │ (None, 2)         │        258 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Formation_GOR_Trend │ (None, 3)         │        387 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Watercut_Trend      │ (None, 3)         │        387 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Oil_Productivity_I… │ (None, 3)         │        387 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 47,124 (184.08 KB)

 Trainable params: 47,124 (184.08 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Split data
# Get the indices of the samples
indices = np.arange(X.shape[0])

# Split the indices into training and validation sets
train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2, # Use 20% of the data for validation
    random_state=SEED # Use the predefined seed for reproducibility
)

# Use the indices to split X and each array in y_oh
X_train = X[train_indices]
X_val = X[val_indices]

y_train = {}
y_val = {}
for col in LABEL_COLS:
    # Get the one-hot encoded array for the current label
    y_oh_col = y_oh[col]
    # Use the indices to split the one-hot encoded array
    y_train[col.replace(' ', '_')] = y_oh_col[train_indices]
    y_val[col.replace(' ', '_')] = y_oh_col[val_indices]

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

X_train shape: (16, 60, 17)
X_val shape: (4, 60, 17)


In [ ]:
'''
# Define the Early Stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)
'''

# Train the model
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=4,
    verbose=2,
    #callbacks=[early_stopping]
)

Epoch 1/50
4/4 - 6s - 2s/step - Formation_GOR_Trend_accuracy: 0.3125 - Formation_GOR_Trend_loss: 1.1327 - Oil_Productivity_Index_accuracy: 0.7500 - Oil_Productivity_Index_loss: 0.7174 - Production_Type_accuracy: 0.6250 - Production_Type_loss: 0.6510 - Reservoir_Name_accuracy: 0.1875 - Reservoir_Name_loss: 1.7695 - Reservoir_Type_accuracy: 0.5625 - Reservoir_Type_loss: 0.6665 - Watercut_Trend_accuracy: 0.3125 - Watercut_Trend_loss: 1.3483 - Well_Type_accuracy: 0.3125 - Well_Type_loss: 0.8298 - loss: 7.1151 - val_Formation_GOR_Trend_accuracy: 0.7500 - val_Formation_GOR_Trend_loss: 1.0106 - val_Oil_Productivity_Index_accuracy: 0.5000 - val_Oil_Productivity_Index_loss: 1.3362 - val_Production_Type_accuracy: 0.2500 - val_Production_Type_loss: 0.9282 - val_Reservoir_Name_accuracy: 0.0000e+00 - val_Reservoir_Name_loss: 1.9189 - val_Reservoir_Type_accuracy: 0.7500 - val_Reservoir_Type_loss: 0.4844 - val_Watercut_Trend_accuracy: 0.7500 - val_Watercut_Trend_loss: 0.8592 - val_Well_Type_accuracy:

In [ ]:
# Plot training history
# Get the history dictionary
history_dict = history.history

# Get the list of epochs
epochs = range(1, len(history_dict['loss']) + 1)

for col in LABEL_COLS:
    # Get the history keys for the current output
    loss_key = col.replace(' ', '_') + '_loss'
    val_loss_key = 'val_' + loss_key
    accuracy_key = col.replace(' ', '_') + '_accuracy'
    val_accuracy_key = 'val_' + accuracy_key

    # Create a new figure for Loss
    fig_loss = go.Figure()
    fig_loss.add_trace(go.Scatter(x=list(epochs), y=history_dict[loss_key], mode='lines', name=f'{col} Training Loss'))
    if val_loss_key in history_dict:
        fig_loss.add_trace(go.Scatter(x=list(epochs), y=history_dict[val_loss_key], mode='lines', name=f'{col} Validation Loss'))
    fig_loss.update_layout(title=f'{col} Training and Validation Loss',
                           xaxis_title='Epochs',
                           yaxis_title='Loss')
    fig_loss.show()

    # Create a new figure for Accuracy
    fig_accuracy = go.Figure()
    fig_accuracy.add_trace(go.Scatter(x=list(epochs), y=history_dict[accuracy_key], mode='lines', name=f'{col} Training Accuracy'))
    if val_accuracy_key in history_dict:
        fig_accuracy.add_trace(go.Scatter(x=list(epochs), y=history_dict[val_accuracy_key], mode='lines', name=f'{col} Validation Accuracy'))
    fig_accuracy.update_layout(title=f'{col} Training and Validation Accuracy',
                               xaxis_title='Epochs',
                               yaxis_title='Accuracy')
    fig_accuracy.show()

In [ ]:
# Evaluate the model using precision, recall and f1 score
# Get the model's predictions on the training data
predictions = model.predict(X)

# Convert the one-hot encoded ground truth labels back to original categorical format
true_labels = {}
for i, col in enumerate(LABEL_COLS):
    true_labels[col] = encoders[col].inverse_transform(np.argmax(y_oh[col], axis=1))

# Convert the model's predicted probabilities to class labels for each output
predicted_labels = {}
for i, col in enumerate(LABEL_COLS):
    predicted_labels[col] = encoders[col].inverse_transform(np.argmax(predictions[i], axis=1))


# Generate and print classification reports for each output
for col in LABEL_COLS:
    print(f"Classification Report for {col}:")
    # Option 2: Use the zero_division parameter
    print(classification_report(true_labels[col], predicted_labels[col], zero_division=0))
    print("-" * 50)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 280ms/step
Classification Report for Reservoir Name:
              precision    recall  f1-score   support

        ACHI       1.00      1.00      1.00         4
        DEPU       0.80      1.00      0.89         4
        JANI       1.00      0.75      0.86         4
        KEMA       0.75      0.75      0.75         4
        MAKO       0.75      0.75      0.75         4

    accuracy                           0.85        20
   macro avg       0.86      0.85      0.85        20
weighted avg       0.86      0.85      0.85        20

--------------------------------------------------
Classification Report for Reservoir Type:
              precision    recall  f1-score   support

   Saturated       0.89      1.00      0.94         8
    Undersat       1.00      0.92      0.96        12

    accuracy                           0.95        20
   macro avg       0.94      0.96      0.95        20
weighted avg       0.96      0.95      0.95        20

----------

In [ ]:
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff

# Iterate through each classification task
for col in LABEL_COLS:
    # Get the true and predicted labels for the current task
    true = true_labels[col]
    predicted = predicted_labels[col]

    # Get the unique classes for the current task to use as labels in the confusion matrix
    classes = encoders[col].classes_

    # Calculate the confusion matrix
    cm = confusion_matrix(true, predicted, labels=classes)

    # Create a Plotly figure for the confusion matrix
    fig = ff.create_annotated_heatmap(
        z=cm,
        x=list(classes),
        y=list(classes),
        colorscale='Viridis',
        showscale=True,
        annotation_text=cm.astype(str)
    )

    # Update layout
    fig.update_layout(
        title=f'Confusion Matrix for {col}',
        xaxis_title='Predicted Label',
        yaxis_title='True Label'
    )

    # Show the plot
    fig.show()

In [ ]:
# Save the well_classifications DataFrame to a CSV file
# Get the unique well names from well_classifications in their current order
unique_well_names_classification = well_classifications.index.unique()

# Create a mapping from old well names to new numerical well names
well_name_mapping_classification = {old_name: f'{i+1}' for i, old_name in enumerate(unique_well_names_classification)}

# Rename well names in well_classifications (index)
well_classifications.index = well_classifications.index.map(well_name_mapping_classification)

# Rename the index name from 'WELL_NAME' to 'Well'
well_classifications.index.name = 'Well'

# Reset the index to make 'Well' a column
well_classifications = well_classifications.reset_index(drop=True)

display(well_classifications)

# Save to CSV, index=False because 'Well' is now a column
well_classifications.to_csv('BDF2M_DSEATS_Africa_2025_Classification.csv', index=False)
print("well_classifications.csv saved successfully.")

,Reservoir Name,Reservoir Type,Well Type,Production Type,Formation GOR Trend,Watercut Trend,Oil Productivity Index
0,JANI,Undersat,GL,Unsteady,aSolGOR,Incr,Decr
1,KEMA,Undersat,NF,Steady,Combo,Combo,Combo
2,MAKO,Saturated,GL,Steady,aSolGOR,Flat,Combo
3,DEPU,Saturated,GL,Steady,aSolGOR,Combo,Combo
4,MAKO,Saturated,NF,Unsteady,aSolGOR,Incr,Flat
5,KEMA,Undersat,NF,Unsteady,aSolGOR,Flat,Combo
6,KEMA,Undersat,GL,Unsteady,aSolGOR,Incr,Combo
7,ACHI,Undersat,GL,Steady,aSolGOR,Incr,Combo
8,DEPU,Saturated,GL,Unsteady,bSolGOR,Incr,Combo
9,JANI,Undersat,NF,Steady,aSolGOR,Flat,Combo


well_classifications.csv saved successfully.


## **Conclusion**

We have successfully tackled the challenge of classifying 20 oil wells based on their production data. We began by loading and cleaning the raw production data, handling data type inconsistencies and missing values. Exploratory Data Analysis provided valuable insights into the well behaviors and data distributions. We then engineered relevant features, including daily production rates, GOR, and watercut, which capture the dynamic nature of well performance. A rule-based classification system was implemented to assign initial labels for key well and reservoir characteristics. Finally, we developed and trained a multi-output convolutional neural network model capable of predicting these multiple classifications simultaneously from time-series production data. The model's performance was evaluated using classification reports and confusion matrices, demonstrating its ability to learn the complex patterns in the data and provide accurate classifications across the different categories. This work provides a robust framework for automating well classification and offers valuable insights into the characteristics and behavior of the wells in the dataset.